In [34]:
import pandas as pd
import re

In [35]:
cps = pd.read_csv(r'cellphones_full.csv')
att = pd.read_csv(r'antutu_score_socket.csv')

In [36]:
cps.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Tên                    966 non-null    str  
 1   Giá                    966 non-null    str  
 2   Link                   966 non-null    str  
 3   Kích thước màn hình    865 non-null    str  
 4   Công nghệ màn hình     806 non-null    str  
 5   Camera sau             847 non-null    str  
 6   Camera trước           817 non-null    str  
 7   Chipset                850 non-null    str  
 8   Công nghệ NFC          763 non-null    str  
 9   Bộ nhớ trong           913 non-null    str  
 10  Thẻ SIM                695 non-null    str  
 11  Hệ điều hành           756 non-null    str  
 12  Độ phân giải màn hình  657 non-null    str  
 13  Tính năng màn hình     725 non-null    str  
 14  Loại CPU               586 non-null    str  
 15  Dung lượng RAM         870 non-null    str  
 16  P

In [37]:
att.info()

<class 'pandas.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Chipset    233 non-null    str  
 1   Antutu_11  233 non-null    int64
 2   Clock      233 non-null    str  
 3   GPU        233 non-null    str  
dtypes: int64(1), str(3)
memory usage: 7.4 KB


In [38]:
_TRASH_VALUES = {
    "",
    "đang cập nhật",
    "mediatek",
    "exynos",
    "snapdragon",
    "bộ xử lý octa-core",
    "asr",
    "asr platform",
    "sc6531e",
    "ums9117",
}
_BRAND_PREFIXES = [
    r"qualcomm\s+(sm|sdm|msm|qm)\w+\s+",   # "Qualcomm SM8350 Snapdragon..." -> "Snapdragon..."
    r"qualcomm\s+",
    r"mediatek\s+",
    r"hisilicon\s+",
    r"samsung\s+",
    r"google\s+",
    r"huawei\s+",
    r"spreadtrum\s+",
    r"unisoc\s+",
    r"apple\s+",
    r"chip\s+",                      
]

def normalize_chipset(text):
    if not isinstance(text, str):
        return ""

    s = text.strip().lower()

    s = re.sub(r"\bthế\s*hệ\b", "gen", s) #"thế hệ" -> "gen"
    s = re.sub(r"\(.*?\)", "", s) #nội dung trong ()
    s = re.sub(r"(\w)\+", r"\1 plus", s) #từ + -> plus
    s = re.sub(r"[®™°•·]", " ", s) #Ký hiệu đặc biệt -> dấu cách
    s = re.sub(r"\b(sm|sdm|msm|apl)\w+\b", "", s) #(sm8350, sdm845, msm8998, apl0698...)

    for pat in _BRAND_PREFIXES:
        s = re.sub(rf"^{pat}", "", s)

    s = re.sub(r"\b(dành cho|cho|danh cho)\s+galaxy\b.*$", "", s)   # "dành cho Galaxy ..."
    s = re.sub(r"\bfor\s+galaxy\b.*$", "", s)                        # "for Galaxy ..."
    s = re.sub(r"\b\d+\s*nhân\b", "", s)                             # "8 nhân", "6 nhân"
    s = re.sub(r"\bocta[\s-]?core\b", "", s)                         # "octa core", "octa-core"
    s = re.sub(r"\b(mobile\s+)?platform\b", "", s)                   # "Mobile Platform"
    s = re.sub(r"\baccelerated\s+edition\b", "", s)                  # "Accelerated Edition"
    s = re.sub(r"\bflagship\b", "", s)                               # "Flagship"
    s = re.sub(r"\btối\s+đa\s+[\d.,]+\s*ghz\b", "", s)              # "tối đa 2.2GHz"
    s = re.sub(r"\btiến\s*trình\b.*$", "", s)                       # "tiến trình 4nm ..."
    s = re.sub(r"\btăng\s+lên\b.*$", "", s)                         # "tăng lên 42% AI ..."
    s = re.sub(r"\b5g\b", "", s)                                     # "5G"
    s = re.sub(r"\b4g\b", "", s)                                     # "4G"

    s = re.sub(r"\b\d+\s*nm\+?\b", "", s) #"6 nm"
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s in _TRASH_VALUES or len(s) < 3:
        return ""

    return s

In [39]:
def map_chipset_info(df_a, df_c) -> pd.DataFrame:

    map = {}
    for _, row in df_a.iterrows():
        norm = normalize_chipset(row["Chipset"])
        map[norm] = {"antutu_11": row["Antutu_11"], "clock": row["Clock"], "gpu": row["GPU"]}
 
    # Map vào từng dòng
    mapped = df_c["Chipset"].apply(lambda x : map.get(normalize_chipset(x)))

    df_out = df_c.copy()
    df_out["antutu_11"] = mapped.apply(lambda x: x["antutu_11"] if x else None)
    df_out["clock"] = mapped.apply(lambda x: x["clock"] if x else None)
    df_out["gpu"] = mapped.apply(lambda x: x["gpu"] if x else None)
 
    return df_out

In [41]:
df = map_chipset_info(att, cps)

In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tên                    966 non-null    str    
 1   Giá                    966 non-null    str    
 2   Link                   966 non-null    str    
 3   Kích thước màn hình    865 non-null    str    
 4   Công nghệ màn hình     806 non-null    str    
 5   Camera sau             847 non-null    str    
 6   Camera trước           817 non-null    str    
 7   Chipset                850 non-null    str    
 8   Công nghệ NFC          763 non-null    str    
 9   Bộ nhớ trong           913 non-null    str    
 10  Thẻ SIM                695 non-null    str    
 11  Hệ điều hành           756 non-null    str    
 12  Độ phân giải màn hình  657 non-null    str    
 13  Tính năng màn hình     725 non-null    str    
 14  Loại CPU               586 non-null    str    
 15  Dung lượng RAM   

In [46]:
df[["Tên", "Chipset", "antutu_11", "clock", "gpu"]].head(30)

,Tên,Chipset,antutu_11,clock,gpu
0,iphone 17,Chip A19 Pro,2606807.0,4260 MHz,Apple A19 Pro GPU
1,oppo find x9s,MediaTek Dimensity 9500s,3041307.0,3730 MHz,Mali-G925 MP12
2,iphone 17 promax,Chip A19 Pro,2606807.0,4260 MHz,Apple A19 Pro GPU
3,samsung galaxy s26,Snapdragon 8 Elite Gen 5 dành cho Galaxy (3nm),3932243.0,4610 MHz,Adreno 840
4,samsung galaxy s26,Exynos 2600 (2nm),3145925.0,3800 MHz,Samsung Xclipse 960
5,samsung galaxy s25,Snapdragon 8 Elite dành cho Galaxy (3nm),3336910.0,4320 MHz,Adreno 830
6,iphone 17,Apple A19,2239708.0,4260 MHz,Apple A19 GPU
7,itel p55,Unisoc T606,330006.0,1600 MHz,Mali-G57 MP1
8,oppo reno15 f,Qualcomm Snapdragon 6 Gen 1 5G,733620.0,2200 MHz,Adreno 710
9,iphone 15,Apple A16 Bionic 6 nhân,1633145.0,3460 MHz,Apple A16 GPU


In [49]:
df['Công nghệ NFC'] = df['Công nghệ NFC'].map(lambda x : 1 if x == "Có" else 0)